# Neural Network from Scratch on Financial Data
## Predicting Monthly Stock Returns using Momentum, Volatility & P/E Ratio

**Architecture:** Input(3) → H1·ReLU(8) → H2·ReLU(4) → Output·Linear(1)

**Key ideas demonstrated:**
- Forward pass: Z = XW + b, A = ReLU(Z)
- Backpropagation: chain rule through each layer
- Mini-batch gradient descent
- Chronological train/test split (no data leakage)
- Gradient inspection across layers

> All core model code is pure numpy — no PyTorch, no sklearn for the network itself.

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
import warnings
warnings.filterwarnings('ignore')

# Our modules
from fetch_data import generate_synthetic_data, clean_and_save
from neural_net import NeuralNet, train_test_split

plt.style.use('dark_background')
COLORS = {
    'cyan':   '#00e5ff',
    'purple': '#a78bfa',
    'green':  '#34d399',
    'red':    '#f87171',
    'amber':  '#f59e0b',
    'grey':   '#475569',
}
print("All imports successful.")

## 2. Data

We use **three cross-sectional features** for each stock-month observation:

| Feature | Description | Academic basis |
|---|---|---|
| Momentum | 12-1 month trailing return | Jegadeesh & Titman (1993) |
| Volatility | 12-month rolling std of returns | Low-vol anomaly |
| P/E Ratio | Price / Earnings | Value factor |

**Target y:** Forward 1-month return (regression)

> If `yfinance` is installed and you have internet, replace `generate_synthetic_data()` with `fetch_real_data()` in fetch_data.py

In [ ]:
# Generate / load data
import os

if os.path.exists("data/features.csv"):
    print("Loading saved data...")
    df = pd.read_csv("data/features.csv")
else:
    print("Generating synthetic financial data...")
    df = generate_synthetic_data(n_samples=500, seed=42)
    df = clean_and_save(df)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Feature & Target Distributions', fontsize=14, color='white', y=1.01)

features_plot = ['momentum', 'volatility', 'pe_ratio', 'target']
feat_colors   = [COLORS['cyan'], COLORS['purple'], COLORS['amber'], COLORS['green']]
feat_labels   = ['Momentum (normalised)', 'Volatility (normalised)',
                 'P/E Ratio (normalised)', 'Target: Next Month Return']

for ax, feat, color, label in zip(axes.flat, features_plot, feat_colors, feat_labels):
    ax.hist(df[feat], bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(df[feat].mean(), color='white', linestyle='--', linewidth=1, label='mean')
    ax.set_title(label, color=color, fontsize=11)
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#475569')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e293b')

fig.patch.set_facecolor('#07090f')
plt.tight_layout()
plt.show()

print("\nDescriptive stats:")
df[features_plot].describe().round(4)

In [ ]:
# Feature-target correlations
print("Pearson correlations with target (next month return):")
for feat in ['momentum', 'volatility', 'pe_ratio']:
    corr = df[feat].corr(df['target'])
    bar  = '█' * int(abs(corr) * 50)
    sign = '+' if corr > 0 else '-'
    print(f"  {feat:>12} : {sign}{abs(corr):.4f}  {bar}")

print("\nNote: Low correlations are expected — monthly returns are mostly noise.")

## 4. Train / Test Split

**Critical:** We use a **chronological split**, not random.

Random splits would leak future observations into training data — the model would appear to work well but fail on real forward-looking predictions. Always split by time in finance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df, test_frac=0.2)

print(f"X_train shape: {X_train.shape}  (n × 3 features)")
print(f"y_train shape: {y_train.shape}  (n × 1 target)")
print(f"X_test  shape: {X_test.shape}")
print(f"y_test  shape: {y_test.shape}")

## 5. Network Architecture

```
Input(3)  →  H1·ReLU(8)  →  H2·ReLU(4)  →  Output·Linear(1)
```

**Parameter count:**

| Layer | Weights | Biases | Total |
|---|---|---|---|
| Input(3) → H1(8) | 3×8 = 24 | 8 | 32 |
| H1(8) → H2(4) | 8×4 = 32 | 4 | 36 |
| H2(4) → Output(1) | 4×1 = 4 | 1 | 5 |
| **Total** | | | **73** |

**Why these sizes?**
- 8 neurons in H1: enough capacity for non-linear interactions between 3 features
- 4 neurons in H2: compression before output
- Linear output: regression — no squashing needed

In [ ]:
model = NeuralNet(layer_sizes=[3, 8, 4, 1], lr=0.01, seed=42)

# Show weight shapes
print("\nWeight matrix shapes:")
for name, param in model.params.items():
    print(f"  {name}: {param.shape}  ({param.size} params)")

## 6. Forward Pass — Single Observation

Before training, let's trace one observation through the network manually to verify shapes.

In [ ]:
# Take first observation
x_single = X_train[0:1]  # shape (1, 3) — keep 2D for matrix multiply

print(f"Input x: {x_single}  shape: {x_single.shape}")
print(f"  momentum={x_single[0,0]:.4f}  volatility={x_single[0,1]:.4f}  pe_ratio={x_single[0,2]:.4f}")

# Forward pass
cache = model.forward(x_single)

print(f"\nLayer-by-layer forward pass:")
print(f"  Z1 = xW1 + b1  →  shape {cache['Z1'].shape}  values: {cache['Z1'].round(4)}")
print(f"  A1 = ReLU(Z1)  →  shape {cache['A1'].shape}  (negatives killed)")
print(f"  Z2 = A1W2 + b2 →  shape {cache['Z2'].shape}  values: {cache['Z2'].round(4)}")
print(f"  A2 = ReLU(Z2)  →  shape {cache['A2'].shape}")
print(f"  Z3 = A2W3 + b3 →  shape {cache['Z3'].shape}  values: {cache['Z3'].round(4)}")
print(f"  ŷ  = Z3 (linear) → {cache['A3'][0,0]:.6f}  (predicted return)")
print(f"\n  Actual y: {y_train[0,0]:.6f}")

## 7. Training — Mini-Batch Gradient Descent

In [ ]:
model = NeuralNet(layer_sizes=[3, 8, 4, 1], lr=0.01, seed=42)
model.train(X_train, y_train, epochs=500, batch_size=32, verbose=True)

## 8. Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#07090f')
ax.set_facecolor('#0d1117')

epochs = range(1, len(model.loss_history) + 1)
ax.plot(epochs, model.loss_history, color=COLORS['cyan'], linewidth=1.5, label='Train MSE')

# Smoothed trend
window = 20
smoothed = pd.Series(model.loss_history).rolling(window).mean()
ax.plot(epochs, smoothed, color=COLORS['amber'], linewidth=2,
        linestyle='--', label=f'{window}-epoch moving avg')

ax.set_xlabel('Epoch', color='#475569')
ax.set_ylabel('MSE Loss', color='#475569')
ax.set_title('Training Loss Curve', color='white', fontsize=13)
ax.legend(facecolor='#0d1117', edgecolor='#1e293b', labelcolor='white')
ax.tick_params(colors='#475569')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e293b')

plt.tight_layout()
plt.show()

print(f"Initial loss : {model.loss_history[0]:.6f}")
print(f"Final loss   : {model.loss_history[-1]:.6f}")
print(f"Reduction    : {(1 - model.loss_history[-1]/model.loss_history[0])*100:.1f}%")

## 9. Evaluation

In [ ]:
print("=" * 45)
train_metrics = model.evaluate(X_train, y_train, label="Train set:")
print()
test_metrics  = model.evaluate(X_test,  y_test,  label="Test set:")
print("=" * 45)

## 10. Predictions vs Actual Returns

In [ ]:
y_pred_test = model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#07090f')

# ── Scatter: predicted vs actual ──────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#0d1117')
ax.scatter(y_test, y_pred_test, color=COLORS['cyan'], alpha=0.5, s=20, edgecolors='none')

# Perfect prediction line
lim = max(abs(y_test.min()), abs(y_test.max()))
ax.plot([-lim, lim], [-lim, lim], color=COLORS['amber'],
        linestyle='--', linewidth=1.5, label='Perfect prediction')

ax.set_xlabel('Actual return', color='#475569')
ax.set_ylabel('Predicted return', color='#475569')
ax.set_title('Predicted vs Actual (Test Set)', color='white')
ax.legend(facecolor='#0d1117', edgecolor='#1e293b', labelcolor='white')
ax.tick_params(colors='#475569')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e293b')

# ── Time series: actual vs predicted ──────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0d1117')
t = range(len(y_test))
ax2.plot(t, y_test,      color=COLORS['green'], linewidth=1.2, label='Actual', alpha=0.9)
ax2.plot(t, y_pred_test, color=COLORS['red'],   linewidth=1.2, label='Predicted', alpha=0.9)
ax2.set_xlabel('Time (test observations)', color='#475569')
ax2.set_ylabel('Return', color='#475569')
ax2.set_title('Actual vs Predicted over Time', color='white')
ax2.legend(facecolor='#0d1117', edgecolor='#1e293b', labelcolor='white')
ax2.tick_params(colors='#475569')
for spine in ax2.spines.values():
    spine.set_edgecolor('#1e293b')

fig.suptitle(f"Test R² = {test_metrics['r2']:.4f}", color=COLORS['amber'], fontsize=12)
plt.tight_layout()
plt.show()

## 11. Gradient Inspection

A healthy network shows similar gradient norms across layers.

- **Vanishing gradients**: early layers show near-zero norms — they learn nothing
- **Exploding gradients**: norms blow up — training becomes unstable
- **Healthy**: norms of similar magnitude across all layers

In [ ]:
# Collect gradient norms across training from scratch
model_inspect = NeuralNet(layer_sizes=[3, 8, 4, 1], lr=0.01, seed=42)

grad_norms = {f"Layer {i}": [] for i in range(1, 4)}
rng = np.random.default_rng(0)

for epoch in range(200):
    idx   = rng.permutation(len(X_train))
    X_bat = X_train[idx[:32]]
    y_bat = y_train[idx[:32]]

    cache = model_inspect.forward(X_bat)
    grads = model_inspect.backward(cache, y_bat)
    model_inspect.update_params(grads)

    for i in range(1, 4):
        grad_norms[f"Layer {i}"].append(np.linalg.norm(grads[f"dW{i}"]))

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#07090f')
ax.set_facecolor('#0d1117')

layer_colors = [COLORS['cyan'], COLORS['purple'], COLORS['green']]
for (layer, norms), color in zip(grad_norms.items(), layer_colors):
    ax.plot(norms, color=color, linewidth=1.5, label=layer, alpha=0.9)

ax.set_xlabel('Epoch', color='#475569')
ax.set_ylabel('||dW|| (gradient norm)', color='#475569')
ax.set_title('Gradient Norms per Layer across Training', color='white', fontsize=13)
ax.legend(facecolor='#0d1117', edgecolor='#1e293b', labelcolor='white')
ax.tick_params(colors='#475569')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e293b')

plt.tight_layout()
plt.show()

print("Final gradient norms (last batch):")
for layer, norms in grad_norms.items():
    print(f"  {layer}: {norms[-1]:.6f}")

## 12. Learning Rate Experiment

How does learning rate affect convergence?

| Learning rate | Effect |
|---|---|
| Too high (0.1) | Loss oscillates or diverges |
| Good (0.01) | Smooth convergence |
| Too low (0.001) | Learns correctly but very slowly |

In [ ]:
lrs     = [0.1, 0.01, 0.001]
lr_cols = [COLORS['red'], COLORS['cyan'], COLORS['purple']]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#07090f')
ax.set_facecolor('#0d1117')

for lr, color in zip(lrs, lr_cols):
    m = NeuralNet(layer_sizes=[3, 8, 4, 1], lr=lr, seed=42)
    m.train(X_train, y_train, epochs=300, batch_size=32, verbose=False)
    ax.plot(m.loss_history, color=color, linewidth=1.5, label=f'lr={lr}', alpha=0.9)

ax.set_xlabel('Epoch', color='#475569')
ax.set_ylabel('MSE Loss', color='#475569')
ax.set_title('Effect of Learning Rate on Convergence', color='white', fontsize=13)
ax.legend(facecolor='#0d1117', edgecolor='#1e293b', labelcolor='white')
ax.tick_params(colors='#475569')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e293b')

plt.tight_layout()
plt.show()

## 13. Backpropagation — Step by Step

Tracing the full chain rule for one mini-batch:

$$\frac{dL}{dW_1} = X^\top \cdot \underbrace{\left(\frac{dL}{dA_2} \odot \text{ReLU}'(Z_1)\right)}_{dL/dZ_1}$$

Where:
$$\frac{dL}{dA_2} = \frac{dL}{dZ_3} \cdot W_3^\top \quad \text{and} \quad \frac{dL}{dZ_3} = \frac{2}{n}(\hat{y} - y)$$

In [ ]:
# Single batch backprop trace
X_bat = X_train[:8]   # 8 observations
y_bat = y_train[:8]

cache = model.forward(X_bat)
grads = model.backward(cache, y_bat)

print("Backprop gradient shapes (must match weight shapes):")
print(f"{'Gradient':>12}  {'Shape':>10}  {'Matches':>10}  {'Norm':>10}")
print("-" * 48)
for i in range(1, 4):
    dw = grads[f"dW{i}"]
    w  = model.params[f"W{i}"]
    db = grads[f"db{i}"]
    b  = model.params[f"b{i}"]
    match = "✓" if dw.shape == w.shape else "✗"
    print(f"  dW{i}:     {str(dw.shape):>10}  {match:>10}  {np.linalg.norm(dw):>10.6f}")
    print(f"  db{i}:     {str(db.shape):>10}  {'✓' if db.shape==b.shape else '✗':>10}  {np.linalg.norm(db):>10.6f}")

print("\nAll gradient shapes match their parameter shapes ✓")

## 14. Honest Interpretation

### Why R² is low — and why that's expected

Monthly stock returns are dominated by noise. The academic literature consistently finds:

- **R² of 0.5–2%** is considered meaningful signal in cross-sectional return prediction
- **Fama & French (1993)** three-factor model explains ~90% of *portfolio* variance but much less for individual stocks
- Our three features (momentum, volatility, P/E) are real academic factors but alone are insufficient

### What this project demonstrates

1. ✅ Full neural network implemented from scratch in numpy
2. ✅ Forward pass with correct matrix dimensions throughout
3. ✅ Backpropagation via chain rule — gradients verified to match weight shapes
4. ✅ Mini-batch gradient descent with chronological splitting
5. ✅ Gradient health inspection across layers
6. ✅ Honest evaluation — low R² acknowledged and explained

### What would improve predictive performance

- More features: earnings revisions, analyst sentiment, sector dummies
- Longer history with real tick data
- Ensemble methods or factor models alongside NN
- Predicting *relative* returns (long-short) rather than absolute

---
> **Note:** The goal of this project is rigorous implementation and understanding — not beating the market.
